# 02. clean

## 0. setup

In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd

# paths
root         = Path.cwd().parent
data_raw     = root / 'data' / 'raw'
data_interim = root / 'data' / 'interim'
data_proc    = root / 'data' / 'processed'

## 1. config

In [2]:
# cartel
in_cartel    = data_raw / 'cartel' / 'manual' / 'cartel_manual_v8.xlsx'
in_nace_isic = data_raw / 'external' / 'nace2_isic4.txt'

# patents
in_pat      = data_interim / 'pat_data.parquet'
in_ipc_isic = data_raw / 'external' / 'ipc4_to_isic_rev4_3_excl_service.txt'    # ALP v2209, excluding services
in_ipc_map  = data_interim / 'ipc_new_to_old.csv'                               # from ipc_map.py


# panel country set: fixed EEA + GB
ref = pd.read_excel(in_cartel, sheet_name='ref_ctry_timeline')
eea = set(ref['ctry_iso'])

# outputs
out_treat = data_interim / 'treat_cartel.parquet'
out_clong = data_interim / 'cartel_long.parquet'
out_scope = data_interim / 'scope_long.parquet'
out_pat   = data_interim / 'pat_isic3.parquet'

print(f'cartel : {in_cartel.name} + {in_nace_isic.name} -> {out_treat.name}, {out_clong.name}')
print(f'patents: {in_pat.name} + {in_ipc_isic.name} -> {out_pat.name}')
print(f'countries: {len(eea)}')

cartel : cartel_manual_v8.xlsx + nace2_isic4.txt -> treat_cartel.parquet, cartel_long.parquet
patents: pat_data.parquet + ipc4_to_isic_rev4_3_excl_service.txt -> pat_isic3.parquet
countries: 31


## 2. cartel

### 2.1 infringements

In [3]:
cart = pd.read_excel(in_cartel, sheet_name='cartels')
n0 = len(cart)

# analysis set: blank exclude_reason
cart = cart[cart['exclude_reason'].isna()]

# industry and decision year must be the same across the segments of an infringement
for c in ['nace_code', 'decision_year']:
    assert (cart.groupby('infringement_id')[c].nunique() == 1).all(), f'{c} varies within infringement'

# collapse segments: one row per infringement
infr = cart.groupby('infringement_id', as_index=False)[['nace_code', 'decision_year']].first()

print(f'cartels: {n0} rows -> {len(cart)} not excluded -> {len(infr)} infringements')

cartels: 218 rows -> 166 not excluded -> 160 infringements


### 2.2 participants

In [4]:
firms = pd.read_excel(in_cartel, sheet_name='firms')
firms.columns = firms.columns.str.strip()
n0 = len(firms)

# keep participants of kept infringements (drops firms of excluded cases) and drop firm_exclude rows
part = firms[firms['infringement_id'].isin(infr['infringement_id']) & firms['firm_exclude'].isna()]
n1 = len(part)

# dates: present and start <= end
assert (part['start_date'] <= part['end_date']).all(), 'missing dates or start > end'

# keep EEA countries
part = part.loc[part['ctry_iso'].isin(eea), ['infringement_id', 'ctry_iso', 'start_date', 'end_date']]

print(f'participants: {n0} rows -> {n1} kept -> {len(part)} in EEA '
      f'| infringements with an EEA participant: {part["infringement_id"].nunique()} / {len(infr)}')

participants: 1825 rows -> 1610 kept -> 1219 in EEA | infringements with an EEA participant: 153 / 160


### 2.3 nace to isic crosswalk

In [5]:
nace_isic = pd.read_csv(in_nace_isic, dtype=str)
nace_isic['nace']  = nace_isic['NACE2code'].str.replace('.', '', regex=False)
nace_isic['isic3'] = nace_isic['ISIC4code'].str[:3]

# split multi-code cells, one row per code
codes = infr[['infringement_id', 'nace_code']].assign(nace=infr['nace_code'].str.split(',')).explode('nace')
codes['nace'] = codes['nace'].str.strip().str[1:]                     # drop section letter: C2932 -> 2932

# 3/4-digit codes: table lookup, never truncation
fine = codes[codes['nace'].str.len() >= 3].merge(nace_isic[['nace', 'isic3']], on='nace', how='left').assign(treat_2d=0)
assert fine['isic3'].notna().all(), f'codes not in crosswalk: {fine.loc[fine["isic3"].isna(), "nace"].tolist()}'

# 2-digit codes: every ISIC3 group in the division, flagged treat_2d = 1
groups = nace_isic.loc[nace_isic['nace'].str.len() == 3, 'isic3'].drop_duplicates()
coarse = (codes[codes['nace'].str.len() == 2]
            .merge(pd.DataFrame({'isic3': groups, 'nace': groups.str[:2]}), on='nace').assign(treat_2d=1))

# infringement x isic3; treat_2d = 1 only if the pair is reached via a 2-digit code alone
imap = pd.concat([fine, coarse]).groupby(['infringement_id', 'isic3'], as_index=False)['treat_2d'].min()

print(f'codes: {len(codes)} ({(codes["nace"].str.len() == 2).sum()} at 2 digits) '
      f'-> {len(imap)} infringement x isic3 pairs | {imap["isic3"].nunique()} isic3 groups')

codes: 188 (15 at 2 digits) -> 213 infringement x isic3 pairs | 59 isic3 groups


### 2.4 annual exposure

In [6]:
# expand each participation to days (start and end included), map to isic3
days = part.assign(day=[pd.date_range(s, e) for s, e in zip(part['start_date'], part['end_date'])]).explode('day')
days = days.merge(imap, on='infringement_id')
days['year'] = days['day'].dt.year

k = ['isic3', 'ctry_iso', 'year']

# cartel_long: infringement x isic3 x country x year (decision cohorts built in 03)
clong = (days[['infringement_id'] + k + ['treat_2d']].drop_duplicates()
           .merge(infr[['infringement_id', 'decision_year']], on='infringement_id'))

# cell-year exposure: distinct cartelised days / days in year (overlapping participants counted once)
treat = days.drop_duplicates(['isic3', 'ctry_iso', 'day']).groupby(k).size().rename('n_days').reset_index()
treat['expo']    = treat['n_days'] / (365 + (treat['year'] % 4 == 0))   # leap years (1969-2022: every 4th)
treat['treated'] = (treat['expo'] >= 0.5).astype(int)                    # main rule; expo > 0 as robustness in 03

# treat_2d = 1 only if every infringement active in the cell-year reaches it via a 2-digit code
treat = treat.merge(clong.groupby(k, as_index=False)['treat_2d'].min(), on=k).drop(columns='n_days')

print(f'cartel_long: {len(clong):,} rows | treat_cartel: {len(treat):,} cell-years with expo > 0, '
      f'{treat["treated"].sum():,} treated | {treat.groupby(["isic3", "ctry_iso"]).ngroups} cells')
print(f'treated via 2-digit only: {treat.loc[treat["treated"] == 1, "treat_2d"].mean():.1%} '
      f'| years {treat["year"].min()}-{treat["year"].max()}')
del days; gc.collect()

cartel_long: 5,551 rows | treat_cartel: 3,655 cell-years with expo > 0, 3,257 treated | 346 cells
treated via 2-digit only: 27.0% | years 1969-2022


28

### 2.5 declared scope

In [7]:
# segments with declared scope (segment dates: scope can change across segments)
seg = cart[['infringement_id', 'ctry_iso', 'infr_start_date', 'infr_end_date']].rename(columns={'ctry_iso': 'code'})
print(f'segments: {len(seg)} | no scope (dropped): {seg.loc[seg["code"].isna(), "infringement_id"].tolist()}')
seg = seg.dropna(subset=['code'])

# one row per segment x code x active year (any-day overlap)
seg = seg.assign(code=seg['code'].str.split(','),
                 year=[list(range(s.year, e.year + 1)) for s, e in zip(seg['infr_start_date'], seg['infr_end_date'])])
seg = seg.explode('code').explode('year')
seg['code'] = seg['code'].str.strip()
seg['year'] = seg['year'].astype(int)

# membership by year: EU = eu_entry <= y < eu_exit; EEA = EU or eea_entry <= y < eea_exit
m   = ref.merge(pd.DataFrame({'year': range(seg['year'].min(), seg['year'].max() + 1)}), how='cross')
eu  = (m['year'] >= m['eu_entry_year']) & ~(m['year'] >= m['eu_exit_year'])
ee  = eu | ((m['year'] >= m['eea_entry_year']) & ~(m['year'] >= m['eea_exit_year']))
mem = pd.concat([m.loc[eu, ['ctry_iso', 'year']].assign(code='EU'),
                 m.loc[ee, ['ctry_iso', 'year']].assign(code='EEA')])

# aggregates -> members in that year; explicit lists literal; keep panel countries
agg = seg['code'].isin(['EU', 'EEA'])
scope = pd.concat([seg[agg].merge(mem, on=['code', 'year']),
                   seg[~agg].rename(columns={'code': 'ctry_iso'})])
scope = scope.loc[scope['ctry_iso'].isin(eea), ['infringement_id', 'ctry_iso', 'year']].drop_duplicates()

# industry: same infringement x isic3 map as treatment (2-digit expansion flagged)
scope = scope.merge(imap, on='infringement_id')[['infringement_id'] + k + ['treat_2d']]

# QA: participant infringement x country pairs outside declared scope (no drop)
pc  = part[['infringement_id', 'ctry_iso']].drop_duplicates()
out = pc.merge(scope[['infringement_id', 'ctry_iso']].drop_duplicates(), how='left', indicator=True)['_merge'].eq('left_only')

# scope cell-years with no participant exposure
cy = scope[k].drop_duplicates().merge(treat[k], how='left', indicator=True)['_merge'].eq('left_only')

print(f'scope_long: {len(scope):,} rows | {scope["infringement_id"].nunique()} infringements | '
      f'{len(cy):,} cell-years, {cy.sum():,} with expo = 0 ({cy.mean():.1%})')
print(f'participant pairs outside declared scope: {out.sum()} / {len(pc)}')

segments: 166 | no scope (dropped): ['AT.32450_i1']
scope_long: 26,855 rows | 159 infringements | 13,846 cell-years, 10,429 with expo = 0 (75.3%)
participant pairs outside declared scope: 80 / 515


### 2.6 save

In [8]:
clong.to_parquet(out_clong, index=False)
treat.to_parquet(out_treat, index=False)
scope.to_parquet(out_scope, index=False)
print(f'saved: {out_clong.name} ({len(clong):,}) | {out_treat.name} ({len(treat):,}) | {out_scope.name} ({len(scope):,})')

saved: cartel_long.parquet (5,551) | treat_cartel.parquet (3,655) | scope_long.parquet (26,855)


## 3. patents

### 3.1 load

In [ ]:
pat = pd.read_parquet(in_pat, columns=['appln_id', 'applt_id', 'ctry_code', 'app_share', 'prio_year', 'ipc', 'cit_fwd_3yr'])
print(f'pat_data: {len(pat):,} rows | {pat["appln_id"].nunique():,} applications | {pat["prio_year"].min()}-{pat["prio_year"].max()}')

### 3.2 applicant weights

In [ ]:
# one row per application x applicant
app = pat[['appln_id', 'applt_id', 'ctry_code', 'app_share']].drop_duplicates(['appln_id', 'applt_id'])
n_all = app['appln_id'].nunique()

# keep EEA applicants with their app_share
app = app[app['ctry_code'].isin(eea)]

print(f'EEA applicants: {app["appln_id"].nunique():,} / {n_all:,} applications | EEA share of mass {app["app_share"].sum() / n_all:.1%}')

### 3.3 ipc4 weights

In [ ]:
alp = pd.read_csv(in_ipc_isic, dtype={'ipc4': str, 'isic_rev4_3': str}).rename(columns={'isic_rev4_3': 'isic3', 'probability_weight': 'w_alp'})

# IPC subclasses created after 2006 (not in ALP) -> ALP predecessors, with shares (from 00_ipc_map.py)
ipc_map = pd.read_csv(in_ipc_map, dtype=str)[['new4', 'old4', 'share']].astype({'share': float})
assert ipc_map['old4'].isin(alp['ipc4']).all()

# IPC4 = first 4 characters; one row per application x distinct IPC4 (EEA-applicant applications only)
ipc = pat.loc[pat['appln_id'].isin(app['appln_id']), ['appln_id', 'prio_year', 'ipc', 'cit_fwd_3yr']].copy()
ipc['ipc4'] = ipc['ipc'].str.replace(' ', '', regex=False).str[:4]
ipc = ipc.drop(columns='ipc').drop_duplicates(['appln_id', 'ipc4'])

# each distinct IPC4 of an application gets 1/n
ipc['w_ipc'] = 1 / ipc.groupby('appln_id')['ipc4'].transform('size')

# new subclasses: replace by predecessor(s), weight split by share (e.g. G16H -> 0.5 G06F, 0.5 G06Q)
ipc = ipc.merge(ipc_map, left_on='ipc4', right_on='new4', how='left')
ipc['w_ipc'] = ipc['w_ipc'] * ipc['share'].fillna(1)
ipc['ipc4']  = ipc['old4'].fillna(ipc['ipc4'])
ipc = ipc.drop(columns=['new4', 'old4', 'share'])

# IPC4 not in ALP are dropped (no renormalisation)
matched = ipc['ipc4'].isin(alp['ipc4'])
print(f'IPC4 mass matched to ALP: {ipc.loc[matched, "w_ipc"].sum() / ipc["appln_id"].nunique():.2%} '
      f'| unmatched: {ipc.loc[~matched, "ipc4"].value_counts().head(5).to_dict()}')
ipc = ipc[matched]

del pat; gc.collect()

### 3.4 assign to industry

In [ ]:
# weight = app_share x 1/n IPC4
rows = app[['appln_id', 'ctry_code', 'app_share']].merge(ipc, on='appln_id')
rows['w']      = rows['app_share'] * rows['w_ipc']
rows['w_cit3'] = rows['w'] * rows['cit_fwd_3yr']

# collapse to country x year x IPC4, then spread over ISIC3 with ALP probabilities
cell = rows.groupby(['ctry_code', 'prio_year', 'ipc4'], as_index=False)[['w', 'w_cit3']].sum().merge(alp, on='ipc4')
cell['pat_frac']  = cell['w'] * cell['w_alp']
cell['cit3_frac'] = cell['w_cit3'] * cell['w_alp']

# outcomes by isic3 x country x year: pat_frac, cit3_frac
panel = (cell.groupby(['isic3', 'ctry_code', 'prio_year'], as_index=False)[['pat_frac', 'cit3_frac']].sum()
             .rename(columns={'ctry_code': 'ctry_iso', 'prio_year': 'year'}))
assert np.isclose(panel['pat_frac'].sum(), rows['w'].sum()), 'mass lost'

print(f'panel: {len(panel):,} cell-years (non-zero only) | {panel["isic3"].nunique()} isic3 | '
      f'{panel["ctry_iso"].nunique()} countries | {panel["year"].min()}-{panel["year"].max()}')
del rows, cell; gc.collect()

### 3.5 diagnostics

In [ ]:
# series by priority year: basis for the sample cutoffs set in 03
by_year = panel.groupby('year')[['pat_frac', 'cit3_frac']].sum()
by_year['cit3_per_pat'] = by_year['cit3_frac'] / by_year['pat_frac']
with pd.option_context('display.max_rows', None):
    print(by_year.round(3).to_string())

### 3.6 save

In [ ]:
panel.to_parquet(out_pat, index=False)
print(f'saved: {out_pat.name} ({len(panel):,} rows)')

del panel, app, ipc; gc.collect()